# Projet 7 : Loan Default Prediction


Lien Kaggle : https://www.kaggle.com/datasets/nikhil1e9/loan-default


About data
This dataset has been taken from Coursera's Loan Default Prediction Challenge.The dataset contains 255,347 rows and 18 columns in total.Our objective is to predict the risk of payment default using machine learning models.

Authors :
BENLARBI Ilias,
SANOGO Fanta,
ATJI Cheick,
WABO Robin


In [1]:
! python.exe -m pip install --upgrade pip
! pip install -r requirements.txt
! pip install xgboost
! pip install imblearn
! pip install ydata_profiling
! pip install shap
! pip install lime

  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
  Using cached pandas-2.2.1-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached matplotlib-3.8.4-cp312-cp312-win_amd64.whl.metadata (5.9 kB)
  Using cached plotly-5.22.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached cufflinks-0.17.3-py3-none-any.whl
  Using cached streamlit-1.41.1-py2.py3-none-any.whl.metadata (8.5 kB)
  Using cached chart_studio-1.1.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached Unidecode-1.3.8-py3-none-any.whl.metadata (13 kB)
  Using cached scikit_learn-1.6.0-cp312-cp312-win_amd64.whl.metadata (15 kB)
  Using cached nltk-3.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached beautifulsoup4-4.12.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached shap-0.46.0-cp312-cp312-win_amd64.whl.metadata (25 kB)
  Using cached mlflow_skinny-2.20.2-py3-none-any.whl.metadata (31 kB)
  Using cached flask-3.1.0-py3-none-any.whl.meta

ERROR: Could not find a version that satisfies the requirement ydata-profiling==4.6.3 (from versions: 4.7.0, 4.8.3, 4.9.0, 4.10.0, 4.11.0, 4.12.0, 4.12.1, 4.12.2, 4.13.0, 4.14.0, 4.15.0, 4.15.1)
ERROR: No matching distribution found for ydata-profiling==4.6.3


## Partie 1 : Analyse exploratoire des données

### Importation des librairies

Commande pour installer les librairies :

- exécuter dans la cellule:  %pip install -r requirements.txt  

ou  

- exécuter dans le terminal (prompt ou powershell):  pip install -r requirements.txt

In [ ]:
import pickle as pkl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import shap
from ydata_profiling import ProfileReport

from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder

from lime import lime_tabular
import warnings
import random
warnings.filterwarnings(action = 'ignore')
random.seed(42)

### Importation des données

In [ ]:
loan_data = pd.read_csv('Loan_default.csv')
loan_data

In [ ]:
# Dimmensions des données
shape=loan_data.shape
print(f"Taille des données: {shape[0]:.2f} lignes\nNombre de variables: {shape[1]}")

loan_data.head(5)

In [ ]:
# Description type de variables
dictionnaire = {
    'LoanID': 'Identifiant unique pour chaque prêt',
    'Age': 'Âge de l’emprunteur',
    'Income': 'Revenu annuel de l’emprunteur',
    'LoanAmount': 'Montant emprunté',
    'CreditScore': 'Score de crédit de l’emprunteur indiquant sa solvabilité',
    'MonthsEmployed': 'Nombre de mois d’emploi de l’emprunteur',
    'NumCreditLines': 'Nombre de lignes de crédit ouvertes par l’emprunteur',
    'InterestRate': 'Taux d’intérêt du prêt',
    'LoanTerm': 'Durée du prêt en mois',
    'DTIRatio': 'Ratio dette/revenu indiquant le niveau d’endettement de l’emprunteur',
    'Education': 'Niveau d’éducation le plus élevé atteint (PhD, Master, Licence, Lycée)',
    'EmploymentType': 'Type de statut d’emploi (Temps plein, Temps partiel, Indépendant, Sans emploi)',
    'MaritalStatus': 'Statut matrimonial de l’emprunteur (Célibataire, Marié, Divorcé)',
    'HasMortgage': 'Indique si l’emprunteur a un prêt hypothécaire (Oui ou Non)',
    'HasDependents': 'Indique si l’emprunteur a des personnes à charge (Oui ou Non)',
    'LoanPurpose': 'But du prêt (Maison, Auto, Éducation, Affaires, Autre)',
    'HasCoSigner': 'Indique si le prêt a un co-emprunteur (Oui ou Non)',
    'Default': 'Variable cible indiquant si le prêt est en défaut (1) ou non (0)'
}


def info(data):

    Information = pd.DataFrame({
        'Variables': data.columns,
        'Type': data.dtypes,
        'Unique_values': data.nunique(),
        'NA_counts': data.isna().sum(),
        'NA_percent%':data.isna().mean().round(4)*100,
        }).reset_index(drop=True)

    Information['Description_des_variables'] = Information['Variables'].map(dictionnaire)

    return Information

# Application sur loan_data
datainfo = info(loan_data)
datainfo

In [ ]:
# Vérification des doublons
datainfo.duplicated().sum()

In [ ]:
# Statistiques descriptives
loan_data.describe()

### Répartition des variables continues

In [ ]:
# création liste des variables continues
variables_continues = ['Age', 'Income', 'LoanAmount', 'CreditScore',
                      'MonthsEmployed', 'NumCreditLines', 'InterestRate',
                      'LoanTerm', 'DTIRatio']

# distribution des variables continues avec Subplots
n_vars = len(variables_continues)
n_cols = 3
n_rows = math.ceil(n_vars / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))

for i, variable in enumerate(variables_continues):
    row = i // n_cols
    col = i % n_cols
    sns.histplot(loan_data[variable], kde=True, ax=axes[row, col])
    axes[row, col].set_title(f'Distribution de {variable}')
    axes[row, col].set_xlabel(variable)
    axes[row, col].set_ylabel('Fréquence')

plt.tight_layout()
plt.show()

In [ ]:
# Création de la figure avec 2 lignes et 6 colonnes
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Liste des colonnes à afficher
columns = [
    'MaritalStatus', 'EmploymentType', 'Education',
    'HasMortgage', 'HasDependents', 'LoanPurpose'
]

# Boucle pour créer les graphiques
for i, col in enumerate(columns):
    row = i // 3
    col_index = i % 3

    loan_data[col].value_counts().plot.pie(ax=axes[row, col_index], autopct='%1.1f%%', startangle=90)
    axes[row, col_index].set_title(f"Répartition {col}")
    axes[row, col_index].set_ylabel("")


plt.tight_layout()
plt.show()

Analyse des résultats :

On observe des distributions relativement uniformes. On voit clairement ici que ce sont des données fictives car elles sont équitablement réparties.

#### Analyse de corrélation avec la méthode de Pearson

In [ ]:
variables_continues = ['Age', 'Income', 'LoanAmount', 'CreditScore',
                      'MonthsEmployed', 'NumCreditLines', 'InterestRate',
                      'LoanTerm', 'DTIRatio']

# Calcul de la matrice de corrélation
correlation_matrix = loan_data[variables_continues].corr()

# Affichage de la matrice de corrélation
print(correlation_matrix)

# Visualisation de la matrice de corrélation avec un heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matrice de corrélation des variables continues")
plt.show()

Les coefficients de corrélation sont très proches de 0. Cela signifie qu'il n'y a pas de forte relation linéaire entre la plupart des variables.

#### Visualisation de la distribution des données avec Boxplot

In [ ]:
variables_continues = ['Age', 'Income', 'LoanAmount', 'CreditScore',
                      'MonthsEmployed', 'NumCreditLines', 'InterestRate',
                      'LoanTerm', 'DTIRatio']

# Calcul du nombre de lignes et de colonnes pour la grille
n_vars = len(variables_continues)
n_cols = 3
n_rows = math.ceil(n_vars / n_cols)

# Création de la grille de sous-graphiques
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows)) # Ajuster la taille si besoin

# Affichage des box plots dans les sous-graphiques
for i, variable in enumerate(variables_continues):
    row = i // n_cols
    col = i % n_cols
    sns.boxplot(x='Default', y=variable, data=loan_data, ax=axes[row, col])
    axes[row, col].set_title(f'Comparaison de {variable} entre les groupes Default')
    axes[row, col].set_xlabel('Default (0: Non, 1: Oui)')
    axes[row, col].set_ylabel(variable)

# Ajustement de l'espacement entre les sous-graphiques
plt.tight_layout()

# Affichage de la grille de sous-graphiques
plt.show()

**Age :**
    On observe une différence notable : une tendance pour les personnes plus jeunes par rapport aux personnes plus agés à être plus susceptibles de faire défaut.


---


**Income (Revenu) :**
    On observe une différence notable : le revenu médian des emprunteurs qui ont fait défaut est inférieur à celui des emprunteurs qui n'ont pas fait défaut. Cela suggère que le revenu est un facteur important dans le risque de crédit.


---


**LoanAmount (Montant du prêt) :**
    On observe que ici que les montants élevés ont tendance à faire défaut.


---


**CreditScore (Score de crédit) :**
    On observe que le score de crédit ne diffère que peu.


---


**MonthsEmployed (Mois d'emploi) :**
    Les emprunteurs qui ont fait défaut semblent avoir une durée d'emploi légèrement inférieure. La stabilité professionnelle est donc un facteur à considérer.


---


**NumCreditLines (Nombre de lignes de crédit) :**
    On observe ici qu'au plus la personne à déjà des crédit, au plus il y a de chance de faire défaut.


---


**InterestRate (Taux d'intérêt) :**
    Les emprunteurs qui ont fait défaut ont un taux d'intérêt plus élevé. Cela est logique, car les prêteurs facturent des taux plus élevés aux emprunteurs considérés comme plus risqués.


---


**LoanTerm (Durée du prêt) :**
    Pas de différence notable


---


**DTIRatio (Ratio dette/revenu) :**
    Pas de différence notable

### Répartition des variables catégorielles


In [ ]:
variables_categorielles = ['Education', 'EmploymentType', 'MaritalStatus',
                           'HasMortgage', 'HasDependents', 'LoanPurpose',
                           'HasCoSigner']

# Calcul du nombre de lignes et de colonnes pour la grille
n_vars = len(variables_categorielles)
n_cols = 3
n_rows = math.ceil(n_vars / n_cols)

# Création de la grille de sous-graphiques
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))

# Affichage des count plots dans les sous-graphiques
for i, variable in enumerate(variables_categorielles):
    row = i // n_cols
    col = i % n_cols
    sns.countplot(x=variable, hue='Default', data=loan_data, ax=axes[row, col])
    axes[row, col].set_title(f'Répartition de {variable} en fonction de Default')
    axes[row, col].set_xlabel(variable)
    axes[row, col].set_ylabel('Fréquence')
    axes[row, col].legend(title='Default', labels=['Non', 'Oui'])

# Suppression des sous-graphiques vides
for j in range(i + 1, n_rows * n_cols):
    fig.delaxes(axes.flatten()[j])

# Ajustement de l'espacement entre les sous-graphiques
plt.tight_layout()

# Affichage de la grille de sous-graphiques
plt.show()

#### Test de Chi-deux

In [ ]:
from scipy.stats import chi2_contingency

variables_categorielles = ['Education', 'EmploymentType', 'MaritalStatus',
                           'HasMortgage', 'HasDependents', 'LoanPurpose',
                           'HasCoSigner']

for variable in variables_categorielles:
    # Création du tableau de contingence
    contingency_table = pd.crosstab(loan_data[variable], loan_data['Default'])

    # Réalisation du test du chi carré
    chi2, p, dof, expected = chi2_contingency(contingency_table)

    # Affichage des résultats
    print(f"Variable : {variable}")
    print(f"Chi2 : {chi2}")
    print(f"P-value : {p}")
    print("-" * 50)

Interprétation des résulats

Toutes les variables catégorielles testées montrent une association significative (p-value <0.05) avec le défaut de paiement. Cela signifie que ces variables peuvent être utilisées comme facteurs prédictifs dans un modèle de risque de crédit

### Répartition de la variable Default

In [ ]:
# Compter le nombre d'observations pour chaque classe
default_counts = loan_data['Default'].value_counts()

# Afficher les résultats
print(default_counts)

# Visualiser la répartition
plt.figure(figsize=(8, 6))
sns.countplot(x='Default', data=loan_data)
plt.title('Répartition de la variable Default')
plt.xlabel('Default (0: Non, 1: Oui)')
plt.ylabel('Fréquence')
plt.show()

Ce graphe montre le déséquilibre suivant :

 La classe "Default = 0" (absence de défaut) est largement majoritaire par rapport à la classe "Default = 1" (défaut de paiement).

Pour le rééquilibrage nous allons partir sur la méthode SMOTE:

Cette méthode consiste à créer de nouvelles observations synthétiques pour la classe minoritaire (Default = 1) afin d'équilibrer le nombre d'observations dans les deux classes, la méthode SMOTE se fait que sur les variables numerique

In [ ]:
loan_data.drop('LoanID',axis = 1 , inplace=True)

loan_data.columns

#### Rééquilibrage avec SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE


variables_categorielles = ['Education', 'EmploymentType', 'MaritalStatus',
                           'HasMortgage', 'HasDependents', 'LoanPurpose',
                           'HasCoSigner']
# encodage one-hot des variables catégorielles
loan_data = pd.get_dummies(loan_data, columns=variables_categorielles, drop_first=True)

#Séparation des variables prédictives et de la variable cible

X = loan_data.drop('Default', axis=1)
y = loan_data['Default']
# application du smote
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

data_resampled = pd.concat([X_resampled, y_resampled], axis=1)

print(data_resampled['Default'].value_counts())

## Partie 2 : Application des modèles

### Train test split

In [ ]:
from sklearn.model_selection import train_test_split

#Séparer les données en features (X) et target (y)
X = data_resampled.drop('Default', axis=1)
y = data_resampled['Default']

#Diviser les données en train et test

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Taille de l'ensemble d'entraînement :", len(X_train))
print("Taille de l'ensemble de test :", len(X_test))

Choix des modèles :

- Arbre de décision (DecisionTreeClassifier) pour l'interprétabilité.

- Random Forest (RandomForestClassifier) pour la robustesse.

- XGBoost (XGBClassifier) pour la performance.

- Validation croisée (cross_val_score()) pour évaluer la stabilité du modèle

### DecisionTreeClassifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialisation du modèle
tree = DecisionTreeClassifier(random_state=42)

# Validation croisée
scores = cross_val_score(tree, X_train, y_train, cv=5, scoring='accuracy')
print("Arbre de décision :")
print(f"  Validation croisée : {scores}")
print(f"  Moyenne : {scores.mean()}")

# Entraînement du modèle
tree.fit(X_train, y_train)

# Prédiction sur l'ensemble de test
y_pred = tree.predict(X_test)

# Évaluation des performances
accuracy = accuracy_score(y_test, y_pred)
print(f"  Accuracy sur l'ensemble de test : {accuracy}")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("-" * 50)

### RandomForestClassifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialisation du modèle
forest = RandomForestClassifier(random_state=42)

# Validation croisée
scores = cross_val_score(forest, X_train, y_train, cv=5, scoring='accuracy')
print("Random Forest:")
print(f"  Validation croisée : {scores}")
print(f"  Moyenne : {scores.mean()}")

# Entraînement du modèle
forest.fit(X_train, y_train)

# Prédiction sur l'ensemble de test
y_pred = forest.predict(X_test)

# Évaluation des performances
accuracy = accuracy_score(y_test, y_pred)
print(f"  Accuracy sur l'ensemble de test : {accuracy}")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("-" * 50)

In [ ]:
X_sample = X_test.sample(n=500, random_state=42)  # 100 lignes au lieu de toutes

explainer = shap.Explainer(tree)
shap_values = explainer.shap_values(X_sample)

shap.summary_plot(shap_values, X_sample)

### XGBClassifier

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialisation du modèle
xgboost = XGBClassifier(random_state=42)

# Validation croisée
scores = cross_val_score(xgboost, X_train, y_train, cv=5, scoring='accuracy')
print("XGBoost :")
print(f"  Validation croisée : {scores}")
print(f"  Moyenne : {scores.mean()}")

# Entraînement du modèle
xgboost.fit(X_train, y_train)

# Prédiction sur l'ensemble de test
y_pred = xgboost.predict(X_test)

# Évaluation des performances
accuracy = accuracy_score(y_test, y_pred)
print(f"  Accuracy sur l'ensemble de test : {accuracy}")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("-" * 50)

In [ ]:
X_sample = X_test.sample(n=500, random_state=42)  # 100 lignes au lieu de toutes

explainer = shap.Explainer(tree)
shap_values = explainer.shap_values(X_sample)

shap.summary_plot(shap_values, X_sample)

## Partie 3 : Performance des modèles

Évaluation des performances de nos modèles

1- Calculer Accuracy(le pourcentage de predictions correctes ), Precision(Proportion de vrais positifs parmi les prédictions positives.), Recall(Proportion de vrais positifs parmi les observations réelles positives.), F1-score(Moyenne harmonique de la précision et du rappel).

2- Tracer une matrice de confusion (confusion_matrix + sns.heatmap) (Matrice de confusion : Tableau qui visualise le nombre de prédictions correctes et incorrectes pour chaque classe).

3-Tracer la courbe ROC-AUC (roc_curve, auc)(Courbe qui représente la performance d'un classificateur binaire à différents seuils de classification. L'AUC (Area Under the Curve) est une mesure de la capacité du classificateur à distinguer les classes.).


### DecisionTreeClassifier

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc

# Prédictions sur l'ensemble de test
y_pred = tree.predict(X_test)

# Calcul des métriques
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Arbre de décision :")
print(f"  Accuracy : {accuracy}")
print(f"  Precision : {precision}")
print(f"  Recall : {recall}")
print(f"  F1-score : {f1}")

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matrice de confusion - Arbre de décision')
plt.xlabel('Prédictions')
plt.ylabel('Réalité')
plt.show()

# Courbe ROC-AUC
y_prob = tree.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Courbe ROC (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de faux positifs')
plt.ylabel('Taux de vrais positifs')
plt.title('Courbe ROC - Arbre de décision')
plt.legend(loc="lower right")
plt.show()

print("-" * 50)

### Random Forest

In [ ]:
# Prédictions sur l'ensemble de test
y_pred = forest.predict(X_test)

# Calcul des métriques
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Forêt aléatoire :")
print(f"  Accuracy : {accuracy}")
print(f"  Precision : {precision}")
print(f"  Recall : {recall}")
print(f"  F1-score : {f1}")

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matrice de confusion - Forêt aléatoire')
plt.xlabel('Prédictions')
plt.ylabel('Réalité')
plt.show()

# Courbe ROC-AUC
y_prob = forest.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Courbe ROC (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de faux positifs')
plt.ylabel('Taux de vrais positifs')
plt.title('Courbe ROC - Forêt aléatoire')
plt.legend(loc="lower right")
plt.show()

print("-" * 50)

### XGBoost

In [ ]:
# Prédictions sur l'ensemble de test
y_pred = xgboost.predict(X_test)

# Calcul des métriques
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("XGBoost :")
print(f"  Accuracy : {accuracy}")
print(f"  Precision : {precision}")
print(f"  Recall : {recall}")
print(f"  F1-score : {f1}")

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matrice de confusion - XGBoost')
plt.xlabel('Prédictions')
plt.ylabel('Réalité')
plt.show()

# Courbe ROC-AUC
y_prob = xgboost.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Courbe ROC (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de faux positifs')
plt.ylabel('Taux de vrais positifs')
plt.title('Courbe ROC - XGBoost')
plt.legend(loc="lower right")
plt.show()

print("-" * 50)

## Partie 4 :  Interpretability & Explicability

### RandomForest model

#### Interprétabilité du modèle

feature_importances = forest.feature_importances_

features_df = pd.DataFrame({
     'Feature': X_train.columns,
    'Importance': feature_importances
})

features_df = features_df.sort_values(by='Importance', ascending=False)
print("Features Contributing to Default Prediction:")
print(features_df.head(10))

plt.figure(figsize=(10,6))
sns.barplot(x='Importance',y='Feature', data=features_df,palette='viridis')
plt.title('Top Features Contributing to Default Prediction of Random Forest')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

#### Explicabilité du modèle

explainer = lime_tabular.LimeTabularExplainer(
                                              training_data = np.array(X_train),
                                              feature_names = X_train.columns.tolist(),
                                              class_names = [0, 1],
                                              verbose = True,
                                              mode='classification'
                                              )

i = 152
exp = explainer.explain_instance(data_row = X_test.iloc[i],
                                 predict_fn=forest.predict_proba,
                                 num_features = 10)

exp.show_in_notebook(show_table=True)

### DecisionTree model

#### Interprétabilité du modèle

X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()


for col in X_train_encoded.select_dtypes(include=['bool']).columns:
    le = LabelEncoder()
    X_train_encoded[col] = le.fit_transform(X_train_encoded[col])


for col in X_test_encoded.select_dtypes(include=['bool']).columns:
    le = LabelEncoder()
    X_test_encoded[col] = le.fit_transform(X_test_encoded[col])

feature_importances = tree.feature_importances_

features_df = pd.DataFrame({
     'Feature': X_train.columns,
    'Importance': feature_importances
})

features_df = features_df.sort_values(by='Importance', ascending=False)
print("Features Contributing to Default Prediction:")
print(features_df.head())

plt.figure(figsize=(10,6))
sns.barplot(x='Importance',y='Feature', data=features_df,palette='viridis')
plt.title('Top Features Contributing to Default Prediction of Decision Tree')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

#### Explicabilité du modèle

explainer = lime_tabular.LimeTabularExplainer(
                                              training_data = np.array(X_train),
                                              feature_names = X_train.columns.tolist(),
                                              class_names = [0, 1],
                                              verbose = True,
                                              mode='classification'
                                              )

i = 152
exp = explainer.explain_instance(data_row = X_test.iloc[i],
                                 predict_fn=tree.predict_proba,
                                 num_features = 10)

exp.show_in_notebook(show_table=True)

### XGBoost model

#### Interprétabilité avec la méthode xgboost

from xgboost import plot_importance
plot_importance(xgboost)
plt.show()

#### Interprétabilité avec la méthode SHAP

explainer = shap.TreeExplainer(xgboost,
                                  feature_names=X_train_encoded.columns.tolist()
                                  )
shap_values = explainer.shap_values(X_train_encoded[:1000])
shap.summary_plot(shap_values, X_train_encoded[:1000])

#### Explicabilité du modèle

explainer = lime_tabular.LimeTabularExplainer(
                                              training_data = np.array(X_train),
                                              feature_names = X_train.columns.tolist(),
                                              class_names = [0, 1],
                                              verbose = True,
                                              mode='classification'
                                              )

i = 152
exp = explainer.explain_instance(data_row = X_test.iloc[i],
                                 predict_fn=xgboost.predict_proba,
                                 num_features = 10)

exp.show_in_notebook(show_table=True)

## Partie 4: Mlflow
- Suivi des performances des Modèles avec Les expériences Mlflow

In [ ]:
import mlflow
from mlflow import MlflowClient

In [ ]:
# lancement du server mlflow en local sur le port 8080
! mlflow server --host 127.0.0.1 --port 8080

- Création d'une nouvelle expérience

In [ ]:
# Connection au server
client = MlflowClient(tracking_uri="http://127.0.0.1:8080")

# Description de l'expérience
experiment_description = ("""
                            Prediction Loan Default machine learning
                            modèle.  
                          """)

# Caractéristiques de l'exécution de l'experience
experiment_tags = {
                    "project_name": "Loan Default Prediction",
                    "team": "Team Fanta-Ilias-Robin-Cheick",
                    "project_quarter": "Q1-2025",
                    "mlflow.note.content": experiment_description,
                    }

# Création de l'expérience: "DecisionTreeclassifier Model"
LoanDefaultExperiment = client.create_experiment(
                                                 name="Loan Default Modeling Experiment", 
                                                 tags=experiment_tags
                                                )

In [ ]:
# nom du modèle
model = tree
params = tree.get_params()


# Nom de l'exécution de l'expérience
name = "decision_tree_run"

# Créer un répertoire "decision_tree_artefact" pour enregistrer 
# tous les fichiers générés par l'expérience decisiontree
artifact_path = "decision_tree_artefact"



with mlflow.start_run(run_name=name) as run:

    y_pred = model.predict(X_test)

    # Calcul des métriques
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1_score = f1_score(y_test, y_pred)

    metrics = {"accuracy": accuracy, 
               "precision": precision, 
               "recall": recall, 
               "f1_score": f1_score}

    # Enregistrer les paramètres utilisés pour entrainer le modele
    mlflow.log_params(params)

    # Enregistrer les métriques
    mlflow.log_metrics(metrics)

    # Log an instance of the trained model for later use
    mlflow.sklearn.log_model(
                              sk_model=model, 
                              input_example=X_test, 
                              artifact_path=artifact_path
                            )
